In [ ]:
!pip install kagglehub --quiet

In [ ]:
import kagglehub
path = kagglehub.dataset_download("mssmartypants/rice-type-classification")


Using Colab cache for faster access to the 'rice-type-classification' dataset.


In [ ]:
print(path)

/kaggle/input/rice-type-classification


In [ ]:
import torch
import torch.nn as nn
from torch.optim import Adam
from torch.utils.data import Dataset, DataLoader
from torchsummary import summary
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from sklearn.preprocessing import MaxAbsScaler
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd


In [ ]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
device

'cuda'

In [ ]:
data_df = pd.read_csv('/kaggle/input/rice-type-classification/riceClassification.csv')
data_df.head()

,id,Area,MajorAxisLength,MinorAxisLength,Eccentricity,ConvexArea,EquivDiameter,Extent,Perimeter,Roundness,AspectRation,Class
0,1,4537,92.229316,64.012769,0.719916,4677,76.004525,0.657536,273.085,0.764510,1.440796,1
1,2,2872,74.691881,51.400454,0.725553,3015,60.471018,0.713009,208.317,0.831658,1.453137,1
2,3,3048,76.293164,52.043491,0.731211,3132,62.296341,0.759153,210.012,0.868434,1.465950,1
3,4,3073,77.033628,51.928487,0.738639,3157,62.551300,0.783529,210.657,0.870203,1.483456,1
4,5,3693,85.124785,56.374021,0.749282,3802,68.571668,0.769375,230.332,0.874743,1.510000,1


In [ ]:
data_df.Class.unique()

array([1, 0])

In [ ]:
data_df.dropna()
data_df.drop(['id'],axis = 1, inplace=True)

In [ ]:
data_df.shape

(18185, 11)

In [ ]:
data_df.head()

,Area,MajorAxisLength,MinorAxisLength,Eccentricity,ConvexArea,EquivDiameter,Extent,Perimeter,Roundness,AspectRation,Class
0,4537,92.229316,64.012769,0.719916,4677,76.004525,0.657536,273.085,0.764510,1.440796,1
1,2872,74.691881,51.400454,0.725553,3015,60.471018,0.713009,208.317,0.831658,1.453137,1
2,3048,76.293164,52.043491,0.731211,3132,62.296341,0.759153,210.012,0.868434,1.465950,1
3,3073,77.033628,51.928487,0.738639,3157,62.551300,0.783529,210.657,0.870203,1.483456,1
4,3693,85.124785,56.374021,0.749282,3802,68.571668,0.769375,230.332,0.874743,1.510000,1


In [ ]:
print(data_df.Class.value_counts())

Class
1    9985
0    8200
Name: count, dtype: int64


In [ ]:
original_data = data_df.copy()


In [ ]:
X = data_df.drop(['Class'],axis =1)
y = data_df['Class']

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X,y, test_size=0.3)

In [ ]:
X_test, X_val , y_test, y_val = train_test_split(X_test,y_test, test_size=0.5)

In [ ]:
scaler = MaxAbsScaler()
X_train = scaler.fit_transform(X_train)
X_val = scaler.transform(X_val)
X_test = scaler.transform(X_test)

In [ ]:
class dataset(Dataset):
  def __init__(self,X,y):
    self.X = torch.tensor(X, dtype= torch.float32).to(device)
    self.y = torch.tensor(y.to_numpy(), dtype= torch.float32).to(device)
  def __len__(self):
    return len(self.X)
  def __getitem__(self,index):
    return self.X[index], self.y[index]


In [ ]:
training_data = dataset(X_train,y_train)
validation_data = dataset(X_val,y_val)
testing_data = dataset(X_test,y_test)

In [ ]:
train_loader = DataLoader(training_data, batch_size=8, shuffle = True)
val_loader = DataLoader(validation_data, batch_size=8, shuffle = True)
test_loader = DataLoader(testing_data, batch_size=8, shuffle = True)

In [ ]:
for x,y in train_loader:
  print(x)
  print("==========")
  print(y)
  break

tensor([[0.5815, 0.7934, 0.6545, 0.9578, 0.5598, 0.7626, 0.5675, 0.6964, 0.7116,
         0.6770],
        [0.8310, 0.8364, 0.8781, 0.9071, 0.7941, 0.9116, 0.7833, 0.7507, 0.8750,
         0.5319],
        [0.6120, 0.7588, 0.7222, 0.9310, 0.5867, 0.7823, 0.7459, 0.6689, 0.8118,
         0.5867],
        [0.8731, 0.9074, 0.8578, 0.9325, 0.8326, 0.9344, 0.6090, 0.7921, 0.8258,
         0.5908],
        [0.5566, 0.7815, 0.6355, 0.9600, 0.5312, 0.7461, 0.6715, 0.6503, 0.7810,
         0.6867],
        [0.8025, 0.8563, 0.8291, 0.9272, 0.7650, 0.8958, 0.7464, 0.7526, 0.8408,
         0.5767],
        [0.6660, 0.9508, 0.6262, 0.9862, 0.6356, 0.8161, 0.8015, 0.7628, 0.6792,
         0.8479],
        [0.7500, 0.8122, 0.8167, 0.9183, 0.7098, 0.8661, 0.6644, 0.7180, 0.8634,
         0.5553]], device='cuda:0')
tensor([1., 0., 0., 0., 1., 0., 1., 0.], device='cuda:0')


In [ ]:
X.shape

(18185, 10)

In [ ]:
hidded_neurons = 10
class myModel(nn.Module):
  def __init__(self):
    super(myModel,self).__init__()
    self.input_layer = nn.Linear(X.shape[1], hidded_neurons)
    self.output_layer = nn.Linear(hidded_neurons,1)
    self.sigmoid = nn.Sigmoid()
  def forward(self, x):
    x = self.input_layer(x)
    x = self.output_layer(x)
    x = self.sigmoid(x)
    return x

In [ ]:
model = myModel().to(device)

In [ ]:
summary(model, (X.shape[1], ))

----------------------------------------------------------------
        Layer (type)               Output Shape         Param #
            Linear-1                   [-1, 10]             110
            Linear-2                    [-1, 1]              11
           Sigmoid-3                    [-1, 1]               0
Total params: 121
Trainable params: 121
Non-trainable params: 0
----------------------------------------------------------------
Input size (MB): 0.00
Forward/backward pass size (MB): 0.00
Params size (MB): 0.00
Estimated Total Size (MB): 0.00
----------------------------------------------------------------
